# 06 — Valuation analysis

**Research question:** How much are investors paying for earnings, sales and cash generation as at **31 December 2025**?

This notebook uses the approved cleaned data and the 2025 provisional financial estimates available on the analysis date. A low multiple is a price statistic, not evidence of an attractive investment. No portfolio decision is made here.

## 1. Inputs and valuation rules

Run from the project root or `notebooks/`. Prices are in pounds, shares in millions, and company financial amounts in £ millions. Thus `price × shares_m` is market value in £ millions. The 2025 closing price is paired with the latest financial record whose `available_date` is no later than 31 December 2025. The 2025 records are provisional estimates known on that date.

A positive P/E is shown only for positive earnings. Negative earnings remain visible through a negative earnings yield. A negative FCF yield records cash consumption rather than a cheap valuation. Multiples with zero or near-zero denominators are missing. Historical comparisons below are **point-in-time year-end snapshots**, using only records then available.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_cleaned_data
from src.research.valuation import valuation_snapshot, annual_valuation_snapshots

annual, prices = load_cleaned_data(project_root / "data" / "Cleaned")
as_of = pd.Timestamp("2025-12-31")
valuation = valuation_snapshot(annual, prices, as_of)
assert len(valuation) == 20 and valuation["ticker"].nunique() == 20
assert valuation["available_date"].le(as_of).all()
print(f"Companies: {len(valuation)}; price date: {valuation['price_date'].iloc[0].date()}")

## ⭐ 2. Valuation measures

| Measure | Formula | Interpretation |
|---|---|---|
| P/E | Market value ÷ positive annual profit | Price paid per £1 of profit; undefined for losses |
| Earnings yield | Annual profit ÷ market value | Signed earnings as a share of price |
| FCF yield | Annual FCF ÷ market value | Signed cash generation as a share of price |
| Dividend yield | Annual dividend per share ÷ share price | Quoted annual dividend relative to price; payment timing is unavailable |
| Price to sales | Market value ÷ positive annual sales or income | Price relative to turnover, with limited cross-sector comparability |

Yields are displayed as percentages. These are snapshot ratios, not forecasts.

In [ ]:
display_columns = ["ticker", "company", "sector", "share_price_gbp", "market_value_gbp_m", "profit_gbp_m", "free_cash_flow_gbp_m", "pe_ratio", "earnings_yield", "fcf_yield", "dividend_yield", "price_to_sales"]
comparison = valuation[display_columns].copy()
comparison = comparison.rename(columns={
    "share_price_gbp": "Price (£)", "market_value_gbp_m": "Market value (£m)",
    "profit_gbp_m": "Profit (£m)", "free_cash_flow_gbp_m": "FCF (£m)",
    "pe_ratio": "P/E (x)", "earnings_yield": "Earnings yield",
    "fcf_yield": "FCF yield", "dividend_yield": "Dividend yield",
    "price_to_sales": "Price/sales (x)"})
formatters = {"Price (£)": "£{:,.2f}", "Market value (£m)": "{:,.0f}",
              "Profit (£m)": "{:,.1f}", "FCF (£m)": "{:,.1f}",
              "P/E (x)": "{:.1f}", "Price/sales (x)": "{:.2f}",
              "Earnings yield": "{:.1%}", "FCF yield": "{:.1%}",
              "Dividend yield": "{:.1%}"}
print("2025 valuation comparison — all 20 companies")
formatted = comparison.copy()
for column, template in formatters.items():
    formatted[column] = formatted[column].map(lambda value: template.format(value) if pd.notna(value) else "n.a.")
display(formatted.reset_index(drop=True))

## ⭐ 3. Independent factsheet check

The **Company Factsheets** (data as at 31 December 2025) quote the following rounded 2025 figures. They are used **only for verification**, never as inputs to calculations. Differences within one unit of the last displayed digit are expected from factsheet rounding. This small check includes a lender and insurer as well as growth and loss-sensitive businesses.

In [ ]:
factsheet_reference = pd.DataFrame([
    # ticker, quoted share price (£), P/E (x), dividend yield, market value (£m)
    ("NCL", 28.75, 23.0, 0.010, 9200),
    ("BFC", 57.09, 45.0, 0.005, 13702),
    ("QCK", 11.73, 110.0, 0.000, 2112),
    ("CRM", 15.17, 12.0, 0.038, 5310),
    ("BSL", 3.00, 6.0, 0.037, 600),
], columns=["ticker", "Factsheet price (£)", "Factsheet P/E (x)",
            "Factsheet dividend yield", "Factsheet market value (£m)"])
checked = factsheet_reference.merge(
    valuation[["ticker", "share_price_gbp", "pe_ratio", "dividend_yield", "market_value_gbp_m"]],
    on="ticker", validate="one_to_one")
checked["Price difference (£)"] = checked["share_price_gbp"] - checked["Factsheet price (£)"]
checked["P/E difference (x)"] = checked["pe_ratio"] - checked["Factsheet P/E (x)"]
checked["Dividend difference (percentage points)"] = 100 * (checked["dividend_yield"] - checked["Factsheet dividend yield"])
checked["Market value difference (£m)"] = checked["market_value_gbp_m"] - checked["Factsheet market value (£m)"]
print("Calculated minus rounded factsheet values")
display(checked[["ticker", "Price difference (£)", "P/E difference (x)",
                 "Dividend difference (percentage points)", "Market value difference (£m)" ]].round(3))
assert checked["Price difference (£)"].abs().max() < 0.01
assert checked["P/E difference (x)"].abs().max() < 0.01
assert checked["Dividend difference (percentage points)"].abs().max() < 0.051
assert checked["Market value difference (£m)"].abs().max() < 1.0

## 4. Sector context and valuation distributions

P/E and price to sales can reflect different business economics across sectors. The following tables and charts describe prices within sectors; they do not identify winners. Companies with losses are excluded from positive P/E summaries but remain in the main table.

In [ ]:
sector_summary = valuation.groupby("sector", dropna=False).agg(
    companies=("ticker", "size"), valid_pe=("pe_ratio", "count"),
    median_pe=("pe_ratio", "median"), median_price_sales=("price_to_sales", "median"),
    median_fcf_yield=("fcf_yield", "median"))
print("Sector valuation summary")
display(sector_summary.round({"median_pe":1, "median_price_sales":2, "median_fcf_yield":3}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sector_summary["median_pe"].sort_values().plot.bar(ax=axes[0], color="#3b6f8e")
axes[0].set(title="Median positive P/E by sector", xlabel="Sector", ylabel="P/E (x)")
valuation["fcf_yield"].dropna().mul(100).plot.hist(ax=axes[1], bins=10, color="#6d8f69", edgecolor="white")
axes[1].axvline(0, color="black", lw=1)
axes[1].set(title="Distribution of FCF yields", xlabel="FCF yield (%)", ylabel="Companies")
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for sector, group in valuation.groupby("sector"):
    ax.scatter(group["price_to_sales"], group["earnings_yield"] * 100, label=sector, s=45, alpha=0.85)
for row in valuation.itertuples():
    if pd.notna(row.price_to_sales) and pd.notna(row.earnings_yield):
        ax.annotate(row.ticker, (row.price_to_sales, row.earnings_yield * 100), xytext=(3, 3),
                    textcoords="offset points", fontsize=7)
ax.axhline(0, color="gray", lw=1)
ax.set(title="Sales multiple versus earnings yield, 2025", xlabel="Price to sales (x)", ylabel="Earnings yield (%)")
ax.legend(title="Sector", fontsize=8)
fig.tight_layout()
plt.show()

## 5. Historical valuation context without look-ahead

At each year end, the calculation selects the latest financial record whose `available_date` has passed. The 2016 annual figures were released in 2017, so there is no 2016 year-end P/E using this workbook. These snapshots use the then-current share count from the selected annual record. They are descriptive comparisons, not a trading backtest.

In [ ]:
historical_valuation = annual_valuation_snapshots(annual, prices)
assert historical_valuation.groupby("valuation_year")["ticker"].nunique().eq(20).all()
print("Year-end P/E context for selected companies")
selected_tickers = ["NCL", "BFC", "QCK", "BSL", "CRM"]
pe_history = historical_valuation.loc[historical_valuation["ticker"].isin(selected_tickers)].pivot(
    index="valuation_year", columns="ticker", values="pe_ratio")
display(pe_history.round(1))
ax = pe_history.plot(figsize=(9, 4.5), marker="o")
ax.set(title="Year-end P/E using financials then available", xlabel="Year end", ylabel="P/E (x)")
ax.grid(alpha=0.25)
plt.show()

## 6. Observations and further checks

- **QCK:** A high positive P/E alongside negative FCF yield means the earnings multiple alone does not describe cash generation. Examine whether its new-city expansion is temporary or structural.
- **BSL:** A low P/E can coexist with rising overdue loans. Credit quality and capital cushion require separate review; the low multiple is a possible value-trap indicator, not proof of one.
- **BFC:** Its higher earnings multiple sets a demanding price context. Compare the implied expectations with cash conversion and contract concentration before drawing conclusions.
- **Sector comparison:** Lender income, insurance premiums and industrial sales have different economics; their price-to-sales ratios are not interchangeable.

**Limitations:** 2025 financials are provisional; historical prices exclude dividends; dividend yield uses annual per-share figures without payment dates; market value uses the reported share count and closing price. The factsheet cross-check confirms selected rounded figures, not every accounting definition. No investment ranking or allocation follows from this notebook.